In [ ]:
import numpy as np
import pandas as pd
from google.colab import files

# ============================================================
# 1. Upload your dataset
# ============================================================
uploaded = files.upload()   # Select your CSV file
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

final_df = df.copy()

# ============================================================
# Helper Functions
# ============================================================

def to_num(series):
    return pd.to_numeric(series, errors="coerce")

def primary_pos(pos):
    if pd.isna(pos):
        return np.nan
    s = str(pos)
    for sep in ["/", "-", ","]:
        if sep in s:
            s = s.split(sep)[0]
            break
    return s.strip()

def z_by_group(df, group_col, value_col):
    x = df[value_col].astype(float)
    g = df[group_col]
    mean = x.groupby(g).transform("mean")
    std  = x.groupby(g).transform("std")
    z = (x - mean) / std
    z = z.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return z

def comp_z(pos_cols, neg_cols=None):
    neg_cols = neg_cols or []
    total = None
    weight = 0.0

    for col in pos_cols:
        if col in final_df.columns:
            z = z_by_group(final_df, "POS_PRIMARY", col)
            total = z if total is None else total + z
            weight += 1.0

    for col in neg_cols:
        if col in final_df.columns:
            z = z_by_group(final_df, "POS_PRIMARY", col)
            total = -z if total is None else total - z
            weight += 1.0

    if total is None or weight == 0:
        return pd.Series(0.0, index=final_df.index)
    return total / weight

# ============================================================
# 2. Determine POS_PRIMARY
# ============================================================

if "POS_nba" in final_df.columns:
    pos_col = "POS_nba"
elif "POS" in final_df.columns:
    pos_col = "POS"
elif "POS_college" in final_df.columns:
    pos_col = "POS_college"
else:
    pos_col = None

if pos_col:
    final_df["POS_PRIMARY"] = final_df[pos_col].apply(primary_pos)
else:
    final_df["POS_PRIMARY"] = "ALL"

# ============================================================
# 3. Convert all numeric fields
# ============================================================

numeric_like = [
    "GP","GS","MP",
    "PTS_nba","TRB_nba","AST_nba","STL_nba","BLK_nba",
    "ORB_nba","DRB_nba","TOV_nba",
    "FG%_nba","3P%_nba","2P%_nba","FT%_nba",
    "TS%_nba","eFG%_nba","TOV%_nba","AST%_nba","USG%_nba",
    "ORB%_nba","DRB%_nba","TRB%_nba",
    "BPM_nba","OBPM_nba","DBPM_nba",
    "OWS_nba","DWS_nba","WS_nba","WS/48",
    "STL%_nba","BLK%_nba",
    "VORP"
]

for col in numeric_like:
    if col in final_df.columns:
        final_df[col] = to_num(final_df[col])

# ============================================================
# 4. Derive Per-Game Stats
# ============================================================

if "GP" in final_df.columns:
    gp = final_df["GP"].replace(0, np.nan)
    for col, newcol in [
        ("PTS_nba","PTS_pg"),
        ("TRB_nba","TRB_pg"),
        ("AST_nba","AST_pg"),
        ("STL_nba","STL_pg"),
        ("BLK_nba","BLK_pg"),
    ]:
        if col in final_df.columns:
            final_df[newcol] = final_df[col] / gp
else:
    final_df["PTS_pg"] = final_df["TRB_pg"] = final_df["AST_pg"] = 0
    final_df["STL_pg"] = final_df["BLK_pg"] = 0

# ============================================================
# 5. Build component scores (uses many stats)
# ============================================================

impact = comp_z(
    pos_cols=[
        "BPM_nba","OBPM_nba","DBPM_nba",
        "WS_nba","WS/48","VORP","OWS_nba","DWS_nba"
    ]
)

scoring = comp_z(
    pos_cols=[
        "PTS_pg","PTS_nba",
        "TS%_nba","eFG%_nba",
        "FG%_nba","3P%_nba","2P%_nba","FT%_nba"
    ],
    neg_cols=["TOV%_nba"]
)

playmaking = comp_z(
    pos_cols=[
        "AST_pg","AST_nba","AST%_nba","USG%_nba"
    ],
    neg_cols=["TOV%_nba"]
)

rebounding = comp_z(
    pos_cols=[
        "TRB_pg","TRB_nba",
        "ORB_nba","DRB_nba",
        "ORB%_nba","DRB%_nba","TRB%_nba"
    ]
)

defense = comp_z(
    pos_cols=[
        "STL_pg","BLK_pg",
        "STL%_nba","BLK%_nba",
        "DBPM_nba","DWS_nba","DRB%_nba"
    ]
)

durability = comp_z(
    pos_cols=["GP","MP","GS"]
)

final_df["_ImpactScore"]  = impact
final_df["_ScoringScore"] = scoring
final_df["_PlayScore"]    = playmaking
final_df["_RebScore"]     = rebounding
final_df["_DefScore"]     = defense
final_df["_DurScore"]     = durability

# Composite weighting
final_df["CompositeScore"] = (
    0.30 * impact +
    0.25 * scoring +
    0.15 * playmaking +
    0.10 * rebounding +
    0.10 * defense +
    0.10 * durability
)

# ============================================================
# 6. Pure percentile-based labels
# ============================================================

valid = final_df["CompositeScore"].astype(float)
valid_non_nan = valid.dropna()

p_super   = np.nanpercentile(valid_non_nan, 95)
p_star    = np.nanpercentile(valid_non_nan, 80)
p_starter = np.nanpercentile(valid_non_nan, 50)
p_role    = np.nanpercentile(valid_non_nan, 20)

def label_from_composite(cs):
    if pd.isna(cs): return "Unknown"
    if cs >= p_super:   return "Superstar"
    if cs >= p_star:    return "Star"
    if cs >= p_starter: return "Starter"
    if cs >= p_role:    return "Role Player"
    return "Bust"

final_df["Label"] = final_df["CompositeScore"].apply(label_from_composite)

# Write labels back into df
df["Label"] = final_df["Label"].values

print("Label distribution:")
print(df["Label"].value_counts())

# ============================================================
# 7. Save & download updated dataset
# ============================================================

output_name = "updated_labeled_dataset.csv"
df.to_csv(output_name, index=False)
files.download(output_name)


Saving final_training_dataset.csv to final_training_dataset (1).csv
Label distribution:
Label
Role Player    360
Starter        360
Bust           240
Star           180
Superstar       60
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>